In [ ]:
import os

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # 0 = all logs, 1 = INFO, 2 = WARNING, 3 = ERROR only
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from tensorflow.keras.datasets.mnist import load_data
(x_train, y_train), (x_test, y_test) = load_data()

In [ ]:
print(x_train)
print('x_train has shape: {}'.format(x_train.shape))

In [ ]:
print(y_train)
print('y_train has shape: {}'.format(y_train.shape))

In [ ]:
norm_x_train = ((x_train - 128.0)/128.0).reshape([-1, 784])

In [ ]:
import numpy as np

def generate_masked_inputs(x, p, seed=None):
    if seed:
        np.random.seed(seed)
    mask = np.random.binomial(n=1, p=p, size=x.shape).astype('float32')
    return x * mask

masked_x_train = generate_masked_inputs(norm_x_train, 0.5)


In [ ]:
from tensorflow.keras import layers, models

autoencoder = models.Sequential([
    layers.Dense(64, activation='relu', input_shape=(784,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(784, activation='tanh')
])

autoencoder.compile(loss='mse', optimizer='adam')
autoencoder.summary()


In [ ]:
history = autoencoder.fit(
    masked_x_train, norm_x_train,
    batch_size=64,
    epochs=10
)

In [ ]:
x_train_sample = x_train[:10]
y_train_sample = y_train[:10]
masked_x_train_sample = generate_masked_inputs(x_train_sample, 0.5, seed=2048)

norm_masked_x = ((x_train - 128.0)/128.0).reshape(-1, 784)
y_pred = autoencoder.predict(norm_masked_x)

print(y_pred)
print('y_pred has shape: {}'.format(y_pred.shape))

In [ ]:
import tensorflow_datasets as tfds
data = tfds.load('cifar10')
print(data)

In [ ]:
import tensorflow as tf
def format_data(x, depth):
    return (tf.cast(x["image"], 'float32'), tf.one_hot(x["label"], depth=depth))


In [ ]:
tr_data = data["train"].map(lambda x: format_data(x, depth=10)).batch(32)

In [ ]:
for d in tr_data.take(1):
    print(d)

In [ ]:
from tensorflow.keras import layers, models
import tensorflow.keras.backend as K

K.clear_session()

cnn = models.Sequential([
    layers.Conv2D(
        filters=16, kernel_size=(9, 9), strides=(2, 2),
        activation='relu', padding='valid',
        input_shape=(32, 32, 3)
    ),
    layers.Conv2D(
        filters=32, kernel_size=(7, 7),
        activation='relu', padding='same'
    ),
    layers.Conv2D(
        filters=64, kernel_size=(7, 7),
        activation='relu', padding='same'
    ),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])


In [ ]:
from tensorflow.keras import layers, models
import tensorflow.keras.backend as K

K.clear_session()

cnn = models.Sequential([
    layers.Conv2D(
        filters=16, kernel_size=(3, 3), strides=(2, 2),
        activation='relu', padding='same',
        input_shape=(32, 32, 3)
    ),
    layers.MaxPool2D(pool_size=(2, 2), strides=(2, 2), padding='same'),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPool2D(pool_size=(2, 2), strides=(2, 2), padding='same'),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(10, activation='softmax')
])


In [ ]:
cnn.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['acc']
)
history = cnn.fit(tr_data,epochs=25)

In [ ]:
import requests
import os

def download_data():
    """
    This function downloads the CO2 data from:
    https://datahub.io/core/co2-ppm/r/co2-mm-gl.csv
    if the file doesn't already exist.
    """
    save_dir = "data"
    save_path = os.path.join(save_dir, 'co2-mm-gl.csv')

    # Create directories if they are not there
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # Download the data and save
    if not os.path.exists(save_path):
        url = "https://datahub.io/core/co2-ppm/r/co2-mm-gl.csv"
        r = requests.get(url)
        with open(save_path, 'wb') as f:
            f.write(r.content)
    else:
        print("co2-mm-gl.csv already exists. Not downloading.")

    return save_path

# Downloading the data
save_path = download_data()

In [ ]:
import pandas as pd

data = pd.read_csv(save_path)

In [ ]:
data.head()

In [ ]:
data = data.set_index('Date')

In [ ]:
data[["Average"]].plot(figsize=(12,6))

In [ ]:
data["Average Diff"] = data["Average"] - data["Average"].shift(1).fillna(method='bfill')
data.head()

In [ ]:
data["Average Diff"].plot(figsize=(12,6))

In [ ]:
import numpy as np

def generate_data(co2_arr, n_seq):
    x, y = [], []
    for i in range(co2_arr.shape[0] - n_seq):
        x.append(co2_arr[i:i + n_seq - 1])
        y.append(co2_arr[i + n_seq - 1:i + n_seq])
    x = np.array(x)
    y = np.array(y)
    return x, y


In [ ]:
from tensorflow.keras import layers, models

rnn = models.Sequential([
    layers.SimpleRNN(64),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

In [ ]:
rnn.compile(loss='mse', optimizer='adam')

In [ ]:
import numpy as np

def generate_data(co2_arr, n_seq):
    x, y = [], []
    for i in range(co2_arr.shape[0] - n_seq):
        x.append(co2_arr[i:i + n_seq - 1])
        y.append(co2_arr[i + n_seq - 1:i + n_seq])
    x = np.array(x).reshape(-1, n_seq - 1, 1)
    y = np.array(y)
    return x, y

In [ ]:
x, y = generate_data(data["Average Diff"].values, n_seq=13)

rnn.fit(
    x, y,
    shuffle=True,
    batch_size=64,
    epochs=25
)

In [ ]:
history = data["Average Diff"].values[-12:].reshape(1, -1, 1)
true_vals = []
prev_true = data["Average"].values[-1]

for i in range(60):
    p_diff = rnn.predict(history).reshape(1, -1, 1)
    history = np.concatenate((history[:, 1:, :], p_diff), axis=1)
    true_vals.append(prev_true + p_diff[0, 0, 0])
    prev_true = true_vals[-1]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Create a datetime index based on monthly frequency for the historical data
dates = pd.date_range(start='1980-01', periods=len(data["Average"]), freq='M')

# Dates for the predicted future values
future_dates = pd.date_range(start=dates[-1] + pd.DateOffset(months=1), periods=len(true_vals), freq='M')

plt.figure(figsize=(10, 6))
plt.plot(dates, data["Average"], 'r--', label='Current trend')
plt.plot(future_dates, true_vals, 'g-', label='Predicted trend')
plt.title("Evolution of CO2 concentration over time")
plt.xlabel("Time")
plt.ylabel("CO2 concentration")
plt.legend()
plt.xticks(rotation=45)
plt.show()
